# 第8讲：决策树、集成学习与模型解释

在订单日预测构件供应批次是否延误，比较树复杂度，并解释随机森林的整体依赖和局部输出。

## 课堂口径

- 一行：一个供应批次的订单日快照；
- 正类：实际到场日晚于需求日期至少1天；
- 主指标：验证集AP；
- 测试集：模型和规则冻结后只评价一次。

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

plt.rcParams['figure.figsize']=(9,4.5)
plt.rcParams['axes.grid']=True
plt.rcParams['grid.alpha']=0.22

## 1. 读取数据并核对时间划分

In [ ]:
MAX_DEPTH=3  # 学生先运行3，再改为7
DATA_PATH=Path('data/prefab_delivery_batches.csv')
SPLIT_PATH=Path('data/time_split.csv')
raw=pd.read_csv(DATA_PATH,parse_dates=['order_date','required_date'])
split_map=pd.read_csv(SPLIT_PATH)
data=raw.merge(split_map[['batch_id','split']],on='batch_id',validate='one_to_one')
data['lead_time_d']=(data['required_date']-data['order_date']).dt.days
summary=data.groupby('split').agg(rows=('batch_id','size'),positives=('target_delivery_late','sum'),positive_rate=('target_delivery_late','mean')).reindex(['train','validation','test'])
display(summary.round(3))
assert len(data)==2200

## 2. 定义特征、模型管道和评价函数

In [ ]:
TARGET='target_delivery_late'
NUMERIC=['component_count','avg_weight_t','transport_distance_km','production_completion_rate','queue_length','supplier_on_time_rate_hist','inspection_first_pass_rate','weather_risk_score','route_congestion_score','change_order_count','lead_time_d']
CATEGORICAL=['component_type']
FEATURES=NUMERIC+CATEGORICAL
train=data[data['split'].eq('train')].copy(); validation=data[data['split'].eq('validation')].copy(); test=data[data['split'].eq('test')].copy()
def make_pipeline(model):
    prep=ColumnTransformer([('num',SimpleImputer(strategy='median'),NUMERIC),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),CATEGORICAL)])
    return Pipeline([('preprocess',prep),('model',model)])
def score_model(pipe,frame):
    score=pipe.predict_proba(frame[FEATURES])[:,1]; pred=(score>=0.5).astype(int); tn,fp,fn,tp=confusion_matrix(frame[TARGET],pred,labels=[0,1]).ravel(); return {'AP':average_precision_score(frame[TARGET],score),'ROC_AUC':roc_auc_score(frame[TARGET],score),'precision':precision_score(frame[TARGET],pred,zero_division=0),'recall':recall_score(frame[TARGET],pred,zero_division=0),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

## 3. 比较一棵树的复杂度

In [ ]:
tree=make_pipeline(DecisionTreeClassifier(max_depth=MAX_DEPTH,min_samples_leaf=10,random_state=2026081308))
tree.fit(train[FEATURES],train[TARGET])
train_result=score_model(tree,train); validation_result=score_model(tree,validation)
leaf_nodes=tree.named_steps['model'].get_n_leaves()
display(pd.DataFrame([{'split':'train',**train_result},{'split':'validation',**validation_result}]).round(3))
print({'max_depth':MAX_DEPTH,'leaf_nodes':leaf_nodes,'train_validation_AP_gap':round(train_result['AP']-validation_result['AP'],3)})
feature_names=tree.named_steps['preprocess'].get_feature_names_out()
tree_importance=pd.DataFrame({'feature':feature_names,'importance':tree.named_steps['model'].feature_importances_}).sort_values('importance',ascending=False)
display(tree_importance.head(8).round(4))

## 4. 学生记录

1. 先保持`MAX_DEPTH=3`运行，记录结构和评价结果；
2. 把`MAX_DEPTH`改成7，重新运行建模单元；
3. 记录`feature_importances_`排名前三的特征；
4. 不要打开测试集结果。

| 最大深度 | 叶节点数 | 训练AP | 验证AP | 差距 | 前三个树内重要特征 |
|---:|---:|---:|---:|---:|---|
| 3 |  |  |  |  |  |
| 7 |  |  |  |  |  |

回答：深度增加后，验证收益是否值得增加的复杂度？树内重要性又不能说明什么？

## Checks

In [ ]:
assert MAX_DEPTH in [3,7]
assert len(train)+len(validation)+len(test)==len(data)
print('结构检查通过；请完成两种深度的比较。')

## Next Steps

1. 比较深度3和7；
2. 用训练—验证差距解释过拟合；
3. 区分全局重要性与单个批次的局部贡献。